In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install mne --quiet
# !pip install autoreject --quiet

import mne
import os, gc, numpy as np, pandas as pd, scipy.signal as sig
# from scipy.signal import hilbert
# from autoreject import AutoReject
import matplotlib.pyplot as plt
from google.colab import files
import plotly.graph_objects as go

# Set path
root_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025/EEGData/'
subject_dirs = sorted([d for d in os.listdir(root_dir) if d.isdigit()])

In [ ]:
# Define 9 regions
regions = {
    'Left Anterior':     ['E48', 'E43', 'E44', 'E38', 'E32', 'E128', 'E25', 'E22', 'E26', 'E23', 'E27', 'E33', 'E34', 'E35', 'E40', 'E39', 'E28', 'E24', 'E20'],
    'Medial Anterior':   ['E12', 'E5', 'E19', 'E11', 'E4', 'E18', 'E16', 'E10', 'E15', 'E21', 'E14', 'E17', 'E127', 'E126'],
    'Right Anterior':    ['E117', 'E118', 'E124', 'E110', 'E109', 'E116', 'E123', 'E3', 'E9', 'E2', 'E122', 'E115', 'E114', 'E121', 'E1', 'E8', 'E125', 'E120', 'E113', 'E119'],
    'Left Central':      ['E29', 'E30', 'E41', 'E45', 'E46', 'E42', 'E47', 'E49', 'E37','E36'],
    'Medial Central':    ['E6', 'E7', 'E13', 'E31','E106', 'E112', 'E80','E54', 'E55', 'E79'],
    'Right Central':     ['E87','E93', 'E98', 'E102', 'E103','E107','E108', 'E104', 'E105','E111'],
    'Left Posterior':    ['E50', 'E51', 'E52', 'E53', 'E56', 'E57', 'E58', 'E59', 'E60','E63', 'E64', 'E65', 'E66', 'E68', 'E69', 'E70', 'E73'],
    'Medial Posterior':  ['E61', 'E62', 'E67', 'E72', 'E77', 'E71', 'E76', 'E75', 'E74', 'E78', 'E82', 'E81'],
    'Right Posterior':   ['E83', 'E84', 'E85', 'E86', 'E88', 'E89', 'E90', 'E91', 'E92','E94', 'E95', 'E96', 'E97', 'E99', 'E100', 'E101']
}

In [ ]:
# SUBJECT 13 - Correlation
subj = '13'
subj_path = os.path.join(root_dir, subj)
session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_001.set')])

# Set correlation threshold
correlation_threshold = 0.6

for session_file in session_files:
    session_path = os.path.join(subj_path, session_file)
    raw = mne.io.read_raw_eeglab(session_path, preload=True)
    raw.set_montage('GSN-HydroCel-128')
    raw.info["bads"] = []

    # Pick EEG channels
    picks = mne.pick_types(raw.info, eeg=True, exclude='bads')
    data, _ = raw[picks]
    ch_names = [raw.ch_names[i] for i in picks]

    # Compute correlation matrix
    corr_matrix = np.corrcoef(data)

    # Calculate average correlation for each channel (excluding self-correlation)
    mean_corrs = []
    for i in range(corr_matrix.shape[0]):
        # Exclude diagonal element (self-correlation)
        mean_corr = (np.sum(corr_matrix[i]) - 1) / (corr_matrix.shape[0] - 1)
        mean_corrs.append(mean_corr)

    # Identify bad channels
    bad_channel_indices = [i for i, avg_corr in enumerate(mean_corrs) if avg_corr < correlation_threshold]
    bad_channels = [ch_names[i] for i in bad_channel_indices]

    # Output
    print(f"\nSession: {session_file}")
    print(f"Total EEG channels: {len(ch_names)}")
    print(f"Correlation threshold: {correlation_threshold}")
    print(f"Detected bad channels (mean corr < {correlation_threshold}):")
    print(bad_channels)
    plt.imshow(corr_matrix)

In [ ]:
# SUBJECT 13 - PLOTS
for subj in subject_dirs:
  if int(subj) == 13:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_001.set')])

    for session_file in session_files:
      session_path = os.path.join(subj_path, session_file)

      # Load
      raw = mne.io.read_raw_eeglab(session_path, preload=True)
      print(f"Processing: {session_path}")

      # Set standard montage
      raw.set_montage('GSN-HydroCel-128')

      raw.info["bads"] = []

      for region_name, region_channels in regions.items():
        picks = [ch for ch in region_channels if ch in raw.ch_names]
        # Filter out missing channels
        if not picks:
          print(f"No valid channels found for {region_name} in this file.")
          continue

        # Add enough dummy channels to reach 20
        dummy_index = 1
        while len(picks) < 20:
          dummy_name = f'dummy{dummy_index}'
          if dummy_name not in raw.ch_names:
            dummy_data = np.zeros((1, raw.n_times))
            dummy_info = mne.create_info([dummy_name], raw.info['sfreq'], ch_types='eeg')
            dummy_raw = mne.io.RawArray(dummy_data, dummy_info)
            raw.add_channels([dummy_raw])
          picks.append(dummy_name)
          dummy_index += 1

        # Plot
        fig=raw.plot(picks=picks, n_channels=20, highpass=0.1, scalings=10e-4, start=0*60, duration=50*60,
          title=f"{session_file} - {region_name}", group_by='original',
          show_scrollbars=False,
          show_scalebars=False)

In [ ]:
# SUBJECT 14 - Correlation
subj = '14'
subj_path = os.path.join(root_dir, subj)
session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_001.set')])

# Set correlation threshold
correlation_threshold = 0.6

for session_file in session_files:
    session_path = os.path.join(subj_path, session_file)
    raw = mne.io.read_raw_eeglab(session_path, preload=True)
    raw.set_montage('GSN-HydroCel-128')
    raw.info["bads"] = []

    # Pick EEG channels
    picks = mne.pick_types(raw.info, eeg=True, exclude='bads')
    data, _ = raw[picks]
    ch_names = [raw.ch_names[i] for i in picks]

    # Compute correlation matrix
    corr_matrix = np.corrcoef(data)

    # Calculate average correlation for each channel (excluding self-correlation)
    mean_corrs = []
    for i in range(corr_matrix.shape[0]):
        # Exclude diagonal element (self-correlation)
        mean_corr = (np.sum(corr_matrix[i]) - 1) / (corr_matrix.shape[0] - 1)
        mean_corrs.append(mean_corr)

    # Identify bad channels
    bad_channel_indices = [i for i, avg_corr in enumerate(mean_corrs) if avg_corr < correlation_threshold]
    bad_channels = [ch_names[i] for i in bad_channel_indices]

    # Output
    print(f"\nSession: {session_file}")
    print(f"Total EEG channels: {len(ch_names)}")
    print(f"Correlation threshold: {correlation_threshold}")
    print(f"Detected bad channels (mean corr < {correlation_threshold}):")
    print(bad_channels)
    plt.imshow(corr_matrix)


In [ ]:
# SUBJECT 14 - PLOTS
for subj in subject_dirs:
  if int(subj) == 14:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_001.set')])

    for session_file in session_files:
      session_path = os.path.join(subj_path, session_file)

      # Load
      raw = mne.io.read_raw_eeglab(session_path, preload=True)
      print(f"Processing: {session_path}")

      # Set standard montage
      raw.set_montage('GSN-HydroCel-128')

      raw.info["bads"] = []

      for region_name, region_channels in regions.items():
        picks = [ch for ch in region_channels if ch in raw.ch_names]
        # Filter out missing channels
        if not picks:
          print(f"No valid channels found for {region_name} in this file.")
          continue

        # Add enough dummy channels to reach 20
        dummy_index = 1
        while len(picks) < 20:
          dummy_name = f'dummy{dummy_index}'
          if dummy_name not in raw.ch_names:
            dummy_data = np.zeros((1, raw.n_times))
            dummy_info = mne.create_info([dummy_name], raw.info['sfreq'], ch_types='eeg')
            dummy_raw = mne.io.RawArray(dummy_data, dummy_info)
            raw.add_channels([dummy_raw])
          picks.append(dummy_name)
          dummy_index += 1

        # Plot
        fig=raw.plot(picks=picks, n_channels=20, scalings='auto', start=40*60, duration=5*60,
          title=f"{session_file} - {region_name}", group_by='original',
          show_scrollbars=False,
          show_scalebars=False)

In [ ]:
# SUBJECT 15 - Correlation
subj = '15'
subj_path = os.path.join(root_dir, subj)
session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_001.set')])

# Set correlation threshold
correlation_threshold = 0.6

for session_file in session_files:
    session_path = os.path.join(subj_path, session_file)
    raw = mne.io.read_raw_eeglab(session_path, preload=True)
    raw.set_montage('GSN-HydroCel-128')
    raw.info["bads"] = []

    # Pick EEG channels
    picks = mne.pick_types(raw.info, eeg=True, exclude='bads')
    data, _ = raw[picks]
    ch_names = [raw.ch_names[i] for i in picks]

    # Compute correlation matrix
    corr_matrix = np.corrcoef(data)

    # Calculate average correlation for each channel (excluding self-correlation)
    mean_corrs = []
    for i in range(corr_matrix.shape[0]):
        # Exclude diagonal element (self-correlation)
        mean_corr = (np.sum(corr_matrix[i]) - 1) / (corr_matrix.shape[0] - 1)
        mean_corrs.append(mean_corr)

    # Identify bad channels
    bad_channel_indices = [i for i, avg_corr in enumerate(mean_corrs) if avg_corr < correlation_threshold]
    bad_channels = [ch_names[i] for i in bad_channel_indices]

    # Output
    print(f"\nSession: {session_file}")
    print(f"Total EEG channels: {len(ch_names)}")
    print(f"Correlation threshold: {correlation_threshold}")
    print(f"Detected bad channels (mean corr < {correlation_threshold}):")
    print(bad_channels)
    plt.imshow(corr_matrix)


In [ ]:
# SUBJECT 15 - PLOTS
for subj in subject_dirs:
  if int(subj) == 15:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_001.set')])

    for session_file in session_files:
      session_path = os.path.join(subj_path, session_file)

      # Load
      raw = mne.io.read_raw_eeglab(session_path, preload=True)
      print(f"Processing: {session_path}")

      # Set standard montage
      raw.set_montage('GSN-HydroCel-128')

      raw.info["bads"] = []

      for region_name, region_channels in regions.items():
        picks = [ch for ch in region_channels if ch in raw.ch_names]
        # Filter out missing channels
        if not picks:
          print(f"No valid channels found for {region_name} in this file.")
          continue

        # Add enough dummy channels to reach 20
        dummy_index = 1
        while len(picks) < 20:
          dummy_name = f'dummy{dummy_index}'
          if dummy_name not in raw.ch_names:
            dummy_data = np.zeros((1, raw.n_times))
            dummy_info = mne.create_info([dummy_name], raw.info['sfreq'], ch_types='eeg')
            dummy_raw = mne.io.RawArray(dummy_data, dummy_info)
            raw.add_channels([dummy_raw])
          picks.append(dummy_name)
          dummy_index += 1

        # Plot
        fig=raw.plot(picks=picks, n_channels=20, scalings='auto', start=40*60, duration=5*60,
          title=f"{session_file} - {region_name}", group_by='original',
          show_scrollbars=False,
          show_scalebars=False)

In [ ]:
# SUBJECT 17 - Correlation
subj = '17'
subj_path = os.path.join(root_dir, subj)
session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_001.set')])

# Set correlation threshold
correlation_threshold = 0.6

for session_file in session_files:
    session_path = os.path.join(subj_path, session_file)
    raw = mne.io.read_raw_eeglab(session_path, preload=True)
    raw.set_montage('GSN-HydroCel-128')
    raw.info["bads"] = []

    # Pick EEG channels
    picks = mne.pick_types(raw.info, eeg=True, exclude='bads')
    data, _ = raw[picks]
    ch_names = [raw.ch_names[i] for i in picks]

    # Compute correlation matrix
    corr_matrix = np.corrcoef(data)

    # Calculate average correlation for each channel (excluding self-correlation)
    mean_corrs = []
    for i in range(corr_matrix.shape[0]):
        # Exclude diagonal element (self-correlation)
        mean_corr = (np.sum(corr_matrix[i]) - 1) / (corr_matrix.shape[0] - 1)
        mean_corrs.append(mean_corr)

    # Identify bad channels
    bad_channel_indices = [i for i, avg_corr in enumerate(mean_corrs) if avg_corr < correlation_threshold]
    bad_channels = [ch_names[i] for i in bad_channel_indices]

    # Output
    print(f"\nSession: {session_file}")
    print(f"Total EEG channels: {len(ch_names)}")
    print(f"Correlation threshold: {correlation_threshold}")
    print(f"Detected bad channels (mean corr < {correlation_threshold}):")
    print(bad_channels)
    plt.imshow(corr_matrix)


In [ ]:
# SUBJECT 17 - PLOTS
for subj in subject_dirs:
  if int(subj) == 15:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_001.set')])

    for session_file in session_files:
      session_path = os.path.join(subj_path, session_file)

      # Load
      raw = mne.io.read_raw_eeglab(session_path, preload=True)
      print(f"Processing: {session_path}")

      # Set standard montage
      raw.set_montage('GSN-HydroCel-128')

      raw.info["bads"] = []

      for region_name, region_channels in regions.items():
        picks = [ch for ch in region_channels if ch in raw.ch_names]
        # Filter out missing channels
        if not picks:
          print(f"No valid channels found for {region_name} in this file.")
          continue

        # Add enough dummy channels to reach 20
        dummy_index = 1
        while len(picks) < 20:
          dummy_name = f'dummy{dummy_index}'
          if dummy_name not in raw.ch_names:
            dummy_data = np.zeros((1, raw.n_times))
            dummy_info = mne.create_info([dummy_name], raw.info['sfreq'], ch_types='eeg')
            dummy_raw = mne.io.RawArray(dummy_data, dummy_info)
            raw.add_channels([dummy_raw])
          picks.append(dummy_name)
          dummy_index += 1

        # Plot
        fig=raw.plot(picks=picks, n_channels=20, scalings='auto', start=40*60, duration=5*60,
          title=f"{session_file} - {region_name}", group_by='original',
          show_scrollbars=False,
          show_scalebars=False)

In [ ]:
# SUBJECT 18 - Correlation
subj = '18'
subj_path = os.path.join(root_dir, subj)
session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_001.set')])

# Set correlation threshold
correlation_threshold = 0.6

for session_file in session_files:
    session_path = os.path.join(subj_path, session_file)
    raw = mne.io.read_raw_eeglab(session_path, preload=True)
    raw.set_montage('GSN-HydroCel-128')
    raw.info["bads"] = []

    # Pick EEG channels
    picks = mne.pick_types(raw.info, eeg=True, exclude='bads')
    data, _ = raw[picks]
    ch_names = [raw.ch_names[i] for i in picks]

    # Compute correlation matrix
    corr_matrix = np.corrcoef(data)

    # Calculate average correlation for each channel (excluding self-correlation)
    mean_corrs = []
    for i in range(corr_matrix.shape[0]):
        # Exclude diagonal element (self-correlation)
        mean_corr = (np.sum(corr_matrix[i]) - 1) / (corr_matrix.shape[0] - 1)
        mean_corrs.append(mean_corr)

    # Identify bad channels
    bad_channel_indices = [i for i, avg_corr in enumerate(mean_corrs) if avg_corr < correlation_threshold]
    bad_channels = [ch_names[i] for i in bad_channel_indices]

    # Output
    print(f"\nSession: {session_file}")
    print(f"Total EEG channels: {len(ch_names)}")
    print(f"Correlation threshold: {correlation_threshold}")
    print(f"Detected bad channels (mean corr < {correlation_threshold}):")
    print(bad_channels)
    plt.imshow(corr_matrix)


In [ ]:
# SUBJECT 18
for subj in subject_dirs:
  if int(subj) == 18:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_001.set')])

    for session_file in session_files:
      session_path = os.path.join(subj_path, session_file)

      # Load
      raw = mne.io.read_raw_eeglab(session_path, preload=True)
      print(f"Processing: {session_path}")

      # Set standard montage
      raw.set_montage('GSN-HydroCel-128')

      raw.info["bads"] = []

      for region_name, region_channels in regions.items():
        picks = [ch for ch in region_channels if ch in raw.ch_names]
        # Filter out missing channels
        if not picks:
          print(f"No valid channels found for {region_name} in this file.")
          continue

        # Add enough dummy channels to reach 20
        dummy_index = 1
        while len(picks) < 20:
          dummy_name = f'dummy{dummy_index}'
          if dummy_name not in raw.ch_names:
            dummy_data = np.zeros((1, raw.n_times))
            dummy_info = mne.create_info([dummy_name], raw.info['sfreq'], ch_types='eeg')
            dummy_raw = mne.io.RawArray(dummy_data, dummy_info)
            raw.add_channels([dummy_raw])
          picks.append(dummy_name)
          dummy_index += 1

        # Plot
        fig=raw.plot(picks=picks, n_channels=20, scalings='auto', start=50*60, duration=5*60,
          title=f"{session_file} - {region_name}", group_by='original',
          show_scrollbars=False,
          show_scalebars=False)



In [ ]:
# SUBJECT 19 - Correlation
subj = '19'
subj_path = os.path.join(root_dir, subj)
session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_001.set')])

# Set correlation threshold
correlation_threshold = 0.6

for session_file in session_files:
    session_path = os.path.join(subj_path, session_file)
    raw = mne.io.read_raw_eeglab(session_path, preload=True)
    raw.set_montage('GSN-HydroCel-128')
    raw.info["bads"] = []

    # Pick EEG channels
    picks = mne.pick_types(raw.info, eeg=True, exclude='bads')
    data, _ = raw[picks]
    ch_names = [raw.ch_names[i] for i in picks]

    # Compute correlation matrix
    corr_matrix = np.corrcoef(data)

    # Calculate average correlation for each channel (excluding self-correlation)
    mean_corrs = []
    for i in range(corr_matrix.shape[0]):
        # Exclude diagonal element (self-correlation)
        mean_corr = (np.sum(corr_matrix[i]) - 1) / (corr_matrix.shape[0] - 1)
        mean_corrs.append(mean_corr)

    # Identify bad channels
    bad_channel_indices = [i for i, avg_corr in enumerate(mean_corrs) if avg_corr < correlation_threshold]
    bad_channels = [ch_names[i] for i in bad_channel_indices]

    # Output
    print(f"\nSession: {session_file}")
    print(f"Total EEG channels: {len(ch_names)}")
    print(f"Correlation threshold: {correlation_threshold}")
    print(f"Detected bad channels (mean corr < {correlation_threshold}):")
    print(bad_channels)
    plt.imshow(corr_matrix)


In [ ]:
# SUBJECT 19-PLOTS
for subj in subject_dirs:
  if int(subj) == 19:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_001.set')])

    for session_file in session_files:
      session_path = os.path.join(subj_path, session_file)

      # Load
      raw = mne.io.read_raw_eeglab(session_path, preload=True)
      print(f"Processing: {session_path}")

      # Set standard montage
      raw.set_montage('GSN-HydroCel-128')

      raw.info["bads"] = []

      for region_name, region_channels in regions.items():
        picks = [ch for ch in region_channels if ch in raw.ch_names]
        # Filter out missing channels
        if not picks:
          print(f"No valid channels found for {region_name} in this file.")
          continue

        # Add enough dummy channels to reach 20
        dummy_index = 1
        while len(picks) < 20:
          dummy_name = f'dummy{dummy_index}'
          if dummy_name not in raw.ch_names:
            dummy_data = np.zeros((1, raw.n_times))
            dummy_info = mne.create_info([dummy_name], raw.info['sfreq'], ch_types='eeg')
            dummy_raw = mne.io.RawArray(dummy_data, dummy_info)
            raw.add_channels([dummy_raw])
          picks.append(dummy_name)
          dummy_index += 1

        # Plot
        fig=raw.plot(picks=picks, n_channels=20, scalings='auto', start=20*60, duration=5*60,
          title=f"{session_file} - {region_name}", group_by='original',
          show_scrollbars=False,
          show_scalebars=False)



In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import mne

# Root EEG directory
root_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025/EEGData'
subject_range = [str(i) for i in range(13, 20)]  # '13' to '19'

# Set correlation threshold
correlation_threshold = 0.6

for subj in subject_range:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_001.set')])

    for session_file in session_files:
        session_path = os.path.join(subj_path, session_file)
        print(f"\n>>> Processing Subject {subj} | Session: {session_file}")

        try:
            # Load EEG data
            raw = mne.io.read_raw_eeglab(session_path, preload=True)
            raw.set_montage('GSN-HydroCel-128')
            raw.info["bads"] = []

            # Pick EEG channels only
            picks = mne.pick_types(raw.info, eeg=True, exclude='bads')
            data, _ = raw[picks]
            ch_names = [raw.ch_names[i] for i in picks]

            # Compute correlation matrix
            corr_matrix = np.corrcoef(data)

            # Compute mean correlation for each channel
            mean_corrs = []
            for i in range(corr_matrix.shape[0]):
                mean_corr = (np.sum(corr_matrix[i]) - 1) / (corr_matrix.shape[0] - 1)
                mean_corrs.append(mean_corr)

            # Find bad channels
            bad_channel_indices = [i for i, avg_corr in enumerate(mean_corrs) if avg_corr < correlation_threshold]
            bad_channels = [ch_names[i] for i in bad_channel_indices]

            # Output
            print(f"  Total channels: {len(ch_names)}")
            print(f"  Correlation threshold: {correlation_threshold}")
            print(f"  BAD channels (mean corr < {correlation_threshold}): {bad_channels}")

            # Plot correlation matrix
            plt.figure(figsize=(10, 8))
            plt.imshow(corr_matrix, cmap='viridis', vmin=-1, vmax=1)
            plt.title(f"Subject {subj} - {session_file} Correlation Matrix")
            plt.colorbar(label='Pearson Correlation')
            plt.tight_layout()
            plt.show()

        except Exception as e:
            print(f"[ERROR] Skipping {subj}/{session_file}: {str(e)}")
